# Hydra的相关配置文件.yaml

## 引入

首先，引入一个代码的例子：
```python
import hydra
CONFIG_PATH = "config/pdm_scoring"
CONFIG_NAME = "default_run_create_submission_pickle"
@hydra.main(config_path=CONFIG_PATH, config_name=CONFIG_NAME, version_base=None)
def main(cfg: DictConfig) -> None:
    """
    Main entrypoint for submission creation script.
    :param cfg: omegaconf dictionary
    """
    agent = instantiate(cfg.agent)
    data_path = Path(cfg.navsim_log_path)
    sensor_blobs_path = Path(cfg.sensor_blobs_path)
    save_path = Path(cfg.output_dir)
    scene_filter = instantiate(cfg.train_test_split.scene_filter)
    ...
```
这里面就用到了Hydra，需要查看Hydra的配置文件：\
通过上面代码就可以得知Hydra的配置文件地址就是：config/pdm_scoring/default_run_create_submission_pickle。这个一个yaml文件：
```python
hydra:
  run:
    dir: ${output_dir}                              #参数	含义   run.dir	指定实验输出目录
  output_subdir: ${output_dir}/code/hydra           # Store hydra's config breakdown here for debugging
  searchpath:                                       # Only <exp_dir> in these paths are discoverable
    - pkg://navsim.planning.script.config.common 
  job:
    chdir: False                  #job.chdir: False	执行任务时不自动切换工作目录

defaults:
  - default_common            #-就相当于是python文件中的list
  - default_evaluation
  - agent: navsim_agent
  - _self_
  - override train_test_split: navmini

"team_name": ke # The team name
"authors": ke_guo # The team members
"email": u3006612@connect.hku.hk # email of the corresponding team member
"institution": ntu # affiliation of the team
"country": Singapore # country or region of the team, e.g. China
```
- 这里的 cfg.sensor_blobs_path 并不是在 Python 文件中定义的，而是从 Hydra 配置对象 cfg 中读取的。


## *1.问题一：sensor_blobs_path = Path(cfg.sensor_blobs_path)，但是上面的yaml文件当中并没有给出来*

- 这里涉及 Hydra 一个非常重要的机制：配置组合（Configuration Composition）
上面的.yaml文件当中拥有：
```python
defaults:
  - default_common
  - default_evaluation
  - agent: navsim_agent
  - _self_
  - override train_test_split: navmini
```
**这相当于告诉 Hydra：**
| 配置项 | 作用 |
| ---- | ---- |
| `default_common` | 引入通用配置 |
| `default_evaluation` | 引入评估相关配置 |
| `agent: navsim_agent` | 引入 Agent 配置 |
| `_self_` | 引入当前 YAML 文件自身的配置 |
| `override train_test_split: navmini` | 将默认的数据集配置替换为 navmini |

*这就说明：sensor_blobs_path 很可能定义在其他配置文件中*
- 这时也就需要我们找到defaults:当中的default_common， default_evaluation

## *2.* 需要yaml文件中引用到的变量

**注意：在上面的yaml配置文件当中有**
```python
searchpath:
  - pkg://navsim.planning.script.config.common
```
这表示 Hydra 还额外添加了一个配置搜索位置：navsim/planning/script/config/common/ \
这时就可以通过下面命令进行寻找
```python
find /home/haha/iPad-main/navsim/planning/script/config 
-name "default_evaluation.yaml"
```

## *3.* 针对yaml中指定的agent: navsim_agent

对于例子中yaml指定的agent: navsim_agent，我们需要去`/home/haha/iPad-main/navsim/planning/script/config/common/agent/navsim_agent.yaml`去寻找\
其中navsim_agent.yaml文件：
```python
_target_: navsim.agents.pad.pad_agent.PadAgent
_convert_: 'all'

config:
  _target_: navsim.agents.pad.navsim_config.PadConfig
  _convert_: 'all'

  trajectory_sampling:
    _target_: nuplan.planning.simulation.trajectory.trajectory_sampling.TrajectorySampling
    _convert_: 'all'
    time_horizon: 4
    interval_length: 0.5

  latent: False

checkpoint_path: /mnt/g/Udata/navsim_exp/ke/navsim/11.07_15.04/lightning_logs/version_0/checkpoints/epoch=0-step=80.ckpt
lr: 1e-4
```
**重点看：** checkpoint_path，这个文件里格式类似于：
```python
checkpoint = {
    "epoch": 0,
    "global_step": 80,
    "state_dict": {
        "agent._pad_model.xxx": tensor(...),
        "agent._pad_model.yyy": tensor(...)
    },
    "optimizer_states": [...]
}
```

# Hydra相关函数

## *1.* instantiate（最重要的函数）

instantiate() 是 Hydra 框架提供的实例化函数,它的作用就是：*根据配置文件中 \_target\_ 指定的类，创建一个 Python 对象。*

例如，你的代码：
```python
worker: WorkerPool = instantiate(cfg.worker)
```
其中 cfg.worker 的配置是：
```python
_target_: navsim.planning.utils.multithreading.worker_ray_no_torch.RayDistributedNoTorch
debug_mode: true
threads_per_node: null
use_distributed: false
```
那么 instantiate(cfg.worker) 就相当于执行：
```python
from navsim.planning.utils.multithreading.worker_ray_no_torch import RayDistributedNoTorch

worker = RayDistributedNoTorch(
    debug_mode=True,
    threads_per_node=None,
    use_distributed=False
)
```
没有在示例中写出的参数，会使用构造函数的默认值。\
它具体进行了三个操作：
1. 找到类：根据 \_target_ 找到 RayDistributedNoTorch。
2. 传入参数：将 YAML 中的配置项传给类的 \_\_init__()。
3. 创建对象：执行 \_\_init__()，将创建好的对象返回给 worker。
注意，你的 RayDistributedNoTorch.\_\_init__() 内部还会调用 initialize()，因此创建对象时也会初始化 Ray。